#Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [4]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

assert WHEELHOUSE_DIR.exists(), (
    f'Wheelhouse not found at {WHEELHOUSE_DIR}. '
    'Attach the wheelhouse dataset in the Input panel (check !ls /kaggle/input/datasets).'
)

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

Installing 41 wheels from /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse...
Wheelhouse installation complete.


## 2. Competition Dataset and Sample Submission

In [5]:
# --- Guard: the install cell must have run first ---
try:
    import swegemma
except ModuleNotFoundError:
    raise RuntimeError(
        "swegemma is not installed. Run the 'Environment Configuration and Package "
        "Installation' cell first, wait for 'Wheelhouse installation complete.', "
        "then rerun this cell."
    )

from swegemma.models import load_tasks
import zipfile
import shutil
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# --- Locate YOUR submission dataset (the attached dataset that contains agent.yaml) ---
# If auto-detection picks the wrong folder, set it by hand, e.g.:
# MY_SUBMISSION_SRC = Path('/kaggle/input/datasets/<owner>/<slug>')
MY_SUBMISSION_SRC = None

if MY_SUBMISSION_SRC is None:
    INPUT_ROOT = Path('/kaggle/input')
    patterns = [
        'datasets/*/*/agent.yaml',
        'datasets/*/*/*/agent.yaml',   # in case it sits inside a wrapper folder
        '*/agent.yaml',
        '*/*/agent.yaml',
    ]
    hits = []
    for pat in patterns:
        hits += [
            p for p in INPUT_ROOT.glob(pat)
            if 'competitions' not in p.parts and 'sample_submission' not in p.parts
        ]

    if hits:
        MY_SUBMISSION_SRC = hits[0].parent
    else:
        # Fallback: the dataset may have kept your files as a .zip
        for z in INPUT_ROOT.glob('datasets/*/*/*.zip'):
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith('agent.yaml') for n in zf.namelist()):
                    extract_dir = Path('/tmp/my_submission_src')
                    shutil.rmtree(extract_dir, ignore_errors=True)
                    zf.extractall(extract_dir)
                    found = list(extract_dir.rglob('agent.yaml'))
                    MY_SUBMISSION_SRC = found[0].parent
                    break

if MY_SUBMISSION_SRC is None:
    listing = sorted(str(p) for p in Path('/kaggle/input').glob('*/*')) + \
              sorted(str(p) for p in Path('/kaggle/input').glob('datasets/*/*'))
    raise FileNotFoundError(
        'Could not find agent.yaml in any attached dataset. '
        'Attach your submission dataset in the Input panel. Found:\n' + '\n'.join(listing)
    )

print(f'Using submission source: {MY_SUBMISSION_SRC}')

# Copy YOUR submission into a writable directory (folder name kept so other cells still work)
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(MY_SUBMISSION_SRC, AGENT_DIR)

# Keep stray metadata out of the scored zip
(AGENT_DIR / 'dataset-metadata.json').unlink(missing_ok=True)

# Sanity checks
assert (AGENT_DIR / 'agent.yaml').exists(), 'agent.yaml is not at the top level of the submission'
assert (AGENT_DIR / 'configs' / 'sampling.yaml').exists(), 'configs/sampling.yaml is missing'

# NOTE: the old code that overwrote configs/sampling.yaml has been removed on purpose.
# Your repo's configs/sampling.yaml is now the source of truth.

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

Using submission source: /kaggle/input/datasets/reidchilds/gemma-agent-submission
Data directory: /kaggle/input/competitions/gemma-4-developer-agent
Agent directory: /kaggle/working/sample_submission
Loaded 129 tasks from tasks.jsonl
  - fastapi_15661 (fastapi/fastapi @ ee22a4b8)
  - fastapi_15588 (fastapi/fastapi @ cb83b83d)
  - fastapi_15589 (fastapi/fastapi @ 22b02e26)
  - fastapi_15030 (fastapi/fastapi @ 48d58ae3)
  - fastapi_14962 (fastapi/fastapi @ 1e78a36b)


## 2.1. Print the file tree

In [6]:
from pathlib import Path

root = Path('/kaggle/working/sample_submission')
assert root.exists(), 'Folder not found. Run the install cell and then the data cell first.'

TEXT_EXT = {'.yaml', '.yml', '.md', '.txt', '.json', '.py', '.toml'}

# File tree with sizes
for p in sorted(root.rglob('*')):
    depth = len(p.relative_to(root).parts) - 1
    size = '' if p.is_dir() else f'  ({p.stat().st_size:,} bytes)'
    print('  ' * depth + ('📁 ' if p.is_dir() else '📄 ') + p.name + size)

# Contents of small text files (config, prompts, skills)
for p in sorted(root.rglob('*')):
    if p.is_file() and p.suffix in TEXT_EXT and p.stat().st_size < 20_000:
        print(f'\n{"=" * 20} {p.relative_to(root)} {"=" * 20}')
        print(p.read_text(errors='replace'))

📁 adapters
  📁 main_lora
    📄 adapter_config.json  (664 bytes)
    📄 adapter_model.safetensors  (217,672 bytes)
  📁 tool_lora
    📄 adapter_config.json  (664 bytes)
    📄 adapter_model.safetensors  (217,672 bytes)
📄 agent.yaml  (438 bytes)
📁 configs
  📄 sampling.yaml  (120 bytes)
📄 eval_config.yaml  (232 bytes)
📁 prompts
  📄 analyzer.md  (527 bytes)
  📄 system.md  (3,888 bytes)
📁 sub_agents
  📄 code_analyzer.yaml  (361 bytes)

==================== adapters/main_lora/adapter_config.json ====================
{
  "alpha_pattern": {},
  "auto_mapping": null,
  "base_model_name_or_path": "google/gemma-4-31b-it-qat-w4a16-ct",
  "bias": "none",
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": [
    0
  ],
  "loftq_config": {},
  "lora_alpha": 8,
  "lora_dropout": 0.0,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "peft_type": "LORA",
 

## 3. Code Graph Functions

In [7]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

Graph for fastapi/fastapi (fastapi_15661):
  Nodes: 4263, Edges: 2430

Neighbors of 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler' (showing up to 5):
  - fastapi/fastapi;custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute
  - fastapi/fastapi;fastapi.exceptions.HTTPException
  - fastapi/fastapi;fastapi.exceptions.RequestValidationError

Top similar nodes to 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler':
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute.get_route_handler (similarity=1.0000)
  - fastapi.routing.APIRouter.__init__ (similarity=0.9996)
  - fastapi.applications.FastAPI.setup (similarity=0.9993)
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute (similarity=0.9991)
  - custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute (similarity=0.9991)

Induced subgraph over 4 focal nodes: 4 nodes, 6 edges


## 4. Start vLLM Server

In [8]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters in the submission directory
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

vLLM server started on http://127.0.0.1:8000/v1 (tp=4)


## 5. Run Phase 1 Inference and Phase 2 Verification

In [9]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read evaluation settings from the submission's eval_config.yaml
eval_config_file = AGENT_DIR / 'eval_config.yaml'
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

# CHANGED: easy-to-edit task count (start at 10; use 2 for a quick smoke test)
NUM_TASKS = 2
SAMPLE_TASKS = tasks[:NUM_TASKS]
limits, gen_constraints = build_submission_limits()

# CHANGED: show what is actually in effect, including any competition-enforced caps
print('eval_config.yaml ->', dict(timeout_seconds=timeout_seconds,
                                  max_tool_calls=max_tool_calls,
                                  max_time_minutes=max_time_minutes,
                                  max_turns=max_turns))
print('limits:', limits)
print('generation constraints:', gen_constraints)

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []
run_log = []  # CHANGED: keep per-task stats for the summary

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    result = run_sync(
        evaluator.evaluate_task,
        task=task,
        task_index=idx,
        total_tasks=len(SAMPLE_TASKS),
    )
    predictions.append({'id': task.instance_id, 'prediction': result.agent_patch or ''})
    run_log.append({
        'id': task.instance_id,
        'repo': task.repo,
        'resolved': bool(result.resolved),
        'patch_chars': len(result.agent_patch or ''),
        'tool_calls': result.tool_calls,
        'duration_s': round(result.duration_seconds, 1),
    })
    print(
        f'  -> resolved={result.resolved}, '
        f'exit_code={result.test_exit_code}, '
        f'patch_chars={len(result.agent_patch or "")}, '
        f'tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])

# CHANGED: summary table to paste into notes/experiments.md
summary_df = pd.DataFrame(run_log)
n_resolved = int(summary_df['resolved'].sum())
n_patched = int((summary_df['patch_chars'] > 0).sum())
print(f'\nResolved {n_resolved}/{len(summary_df)} | '
      f'non-empty patches {n_patched}/{len(summary_df)} | '
      f'avg tool calls {summary_df["tool_calls"].mean():.1f} | '
      f'avg duration {summary_df["duration_s"].mean():.1f}s')
display(summary_df)

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.AGENT_CONFIG is enabled.
  check_feature_enabled()
/tmp/ipykernel_163/3494609774.py:65: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(


eval_config.yaml -> {'timeout_seconds': 60, 'max_tool_calls': 10, 'max_time_minutes': 1.0, 'max_turns': 50}
limits: SubmissionLimits(max_total_size_bytes=3221225472, max_yaml_size_bytes=52428800, max_skill_size_bytes=52428800, max_file_count=10000, max_yaml_files=1000, max_instruction_chars=1000000, max_total_instruction_chars=10000000, max_agents=500, max_sub_agent_depth=50, max_loop_iterations=500, max_skills=1000, allowed_file_extensions=frozenset({'.txt', '.json', '.safetensors', '.md', '.py', '.yaml', '.yml'}), adapter_extensions=frozenset({'.safetensors'}))
generation constraints: GenerationConstraints(allowed_fields=None, temperature=None, top_p=None, top_k=None, max_output_tokens=NumericRange(min=1, max=32768), presence_penalty=None, frequency_penalty=None, thinking_budget=NumericRange(min=0, max=32768), max_stop_sequences=None, allowed_response_mime_types=None, defaults={'max_output_tokens': 16384, 'thinking_config': {'thinking_budget': 4096}})
[1/2] Evaluating fastapi_15661 (

[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [  19.7s] 💭 (986 chars) "The user wants to "Automate release preparation".
Looking at the problem statement, it seems like a PR description rathe..."
  [  19.7s] 🔧 run_command find . -maxdepth 2 -name "*release*"
  [  19.8s]   → ./scripts/add_latest_release_date.py
  [  21.5s] 💭 (86 chars) "Okay, there is `./scripts/add_latest_release_date.py`.
Let's see what's in `scripts/`."
  [  21.5s] 🔧 run_command ls scripts/
  [  21.5s]   → add_latest_release_date.py
  [  25.6s] 💭 (338 chars) "The problem statement is very vague. "Automate release preparation".
If I look at the `asyncer` PR 608 (if I could), I m..."
  [  25.6s] 📖 read_file (scripts/add_latest_release_date.py)
  [  25.7s]   ✓ {'status': 'ok', 'filepath': 'scripts/add_latest_release_date.py', 'content': '"""Check 
release-notes.md and add today\'…
  [  31.3s] 💭 (530 chars) "This script just adds the date to the release notes.

The problem statement says "Automate release preparation".
Maybe i..."
  [  31.3s] 🔧 run_command grep -r "release" . | grep ".py" | head -n 20
  [  31.4s]   → ./scripts/add_latest_release_date.py:"""Check release-notes.md and add today's date to the latest 
release header if miss
  [  36.9s] 💭 (479 chars) "I see `.pre-commit-config.yaml` has `uv run python scripts/add_latest_release_date.py`.

If the task is to "Automate rel..."
  [  36.9s] 📖 read_file (pyproject.toml)
  [  36.9s]   ✓ {'status': 'ok', 'filepath': 'pyproject.toml', 'content': '[build-system]\nrequires = 
["pdm-backend"]\nbuild-backend = "…
  [  40.4s] 💭 (147 chars) "`dynamic = ["version"]` in `pyproject.toml`. This means the version is likely defined 
in `fastapi/__init__.py`.

Let's c..."
  [  40.4s] 📖 read_file (fastapi/__init__.py)
  [  40.5s]   ✓ {'status': 'ok', 'filepath': 'fastapi/__init__.py', 'content': '"""FastAPI framework, high 
performance, easy to learn, f…
  [  49.5s] 💭 (861 chars) "Yes, `__version__ = "0.136.3"`.

Now, what does "Automate release preparation" mean?
If I look at the `asyncer` PR 608, ..."
  [  49.5s] 🔧 run_command grep -r "asyncer" .
  [  49.7s]   → ./fastapi/.agents/skills/fastapi/references/other-tools.md:uv add asyncer

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   🔧 run_command                                                                                                │
│     ... (11 earlier lines)                                                                                      │
│     ./docs/zh-hant/docs/async.md:我另外在 AnyIO                                                                 │
│ 之上做了一個薄封裝的函式庫，稍微改進型別註解以獲得更好的**自動補全**、**即時錯誤**等。同時它也提供友善的介紹與  │
│ 教學，幫助你**理解**並撰寫**自己的非同步程式碼**：[Asyncer](https://asyncer.tiangolo.com/)。當你需要**將非同步  │
│ 程式碼與一般**（阻塞/同步）**程式碼整合**時，它特別實用。                                                       │
│     ./docs/fr/docs/advanced/stream-data.md:Si vous devez appeler du code bloquant depuis une fonction async, ou │
│ une fonction async depuis une fonction bloquante, vous pouvez utiliser [Asyncer](https://asyncer.tiangolo.com), │
│ une bibliothèque sœur de FastAPI.                                                                               │
│     ./docs/fr/docs/async.md:J’ai créé une autre bibliothèque au-dessus d’AnyIO, comme une fine surcouche, pour  │
│ améliorer un peu les annotations de type et obtenir une meilleure **autocomplétion**, des **erreurs en ligne**, │
│ etc. Elle propose également une introduction et un tutoriel accessibles pour vous aider à **comprendre** et     │
│ écrire **votre propre code async** : [Asyncer](https://asyncer.tiangolo.com/). Elle sera particulièrement utile │
│ si vous devez **combiner du code async avec du code classique** (bloquant/synchrone).          

  -> resolved=False, exit_code=-1, patch_chars=0, tool_calls=7, duration=65.6s
[2/2] Evaluating fastapi_15588 (fastapi/fastapi)...


[   0.0s] 📋 System Prompt: You are an expert autonomous software engineer assigned to resolve an issue in a 
repository efficiently and decisively.
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [   5.1s] 💭 (284 chars) "The user wants to validate Server Sent Event (SSE) fields in FastAPI to prevent sending
broken data (specifically multi-..."
  [   5.1s] 🤖 search_similar_code {"query": "Server Sent Events"}
  [   6.3s]   → search_similar_code: {"status": "ok", "query": "Server Sent Events", "results": [], "count": 0, 
"is_truncated": false}
  [  11.2s] 💭 (524 chars) "Maybe it's called `EventSource` or something similar. Let me try searching for 
"EventSource" or "SSE".
Actually, FastAPI..."
  [  11.2s] 🔧 run_command grep -r "SSE" .
  [  11.3s]   → ./fastapi/.agents/skills/fastapi/references/streaming.md:## Server-Sent Events (SSE)
  [  14.9s] 💭 (147 chars) "The `grep` results point to `fastapi/sse.py` as a key file. It mentions 
`ServerSentEvent` and `_SSE_EVENT_SCHEMA`.

Let'..."
  [  14.9s] 📖 read_file (fastapi/sse.py)
  [  14.9s]   ✓ {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': 'from typing import Annotated, 
Any\n\nfrom annotated_doc impor…

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   📖 read_file                                                                                                  │
│                                                                                                                 │
│   Path:                                                                                                         │
│     fastapi/sse.py                                                                                              │
│                                                                                                                 │
│   Response:                                                                                                     │
│     {'status': 'ok', 'filepath': 'fastapi/sse.py', 'content': 'from typing import Annotated, Any\n\nfrom        │
│ annotated_doc import Doc\nfrom pydantic import AfterValidator, BaseModel, Field, model_validator\nfrom          │
│ starlette.responses import StreamingResponse\n\n# Canonical SSE event schema matching the OpenAPI 3.2 spec\n#   │
│ (Section 4.14.4 "Special Considerations for Server-Sent Events")\n_SSE_EVENT_SCHEMA: dict[str, Any] = {\n       │
│ "type": "object",\n    "properties": {\n        "data": {"type": "string"},\n        "event": {"type":          │
│ "string"},\n        "id": {"type": "string"},\n        "retry": {"type": "integer", "minimum": 0},\n            │
│ },\n}\n\n\nclass EventSourceResponse(StreamingResponse):\n    """Streaming response with `text/event-stream`    │
│ media type.\n\n    Use as `response_class=EventSourceResponse` on a *path operation* that uses `yield`\n    to  │
│ enable Server Sent Events (SSE) responses.\n\n    Works with **any HTTP method** (`GET`, `POST`, etc.), which   │
│ makes it compatible\n    with protocols like MCP that stream SSE over `POST`.\n\n    The actual encoding logic  │
│ lives in the FastAPI routing layer. This class\n    serves mainly as a marker and sets the correct              │
│ `Content-Type`.\n    """\n\n    media_type = "text/event-stream"\n\n\ndef _check_id_no_null(v: str | None) ->   │
│ str | None:\n    if v is not None and "\\0" in v:\n        raise ValueError("SSE \'id\' must not contain null   │
│ characters")\n    return v\n\n\nclass ServerSentEvent(BaseModel):\n    """Represents a single Server-Sent       │
│ Event.\n\n    When `yield`ed from a *path operation function* that uses\n                                       │
│ `response_class=EventSourceResponse`, each `ServerSentEvent` is encoded\n    into the [SSE wire                 │
│ format](https://html.spec.whatwg.org/multipage/server-sent-events.html#parsing-an-event-strea

  -> resolved=False, exit_code=-1, patch_chars=0, tool_calls=3, duration=66.2s

Resolved 0/2 | non-empty patches 0/2 | avg tool calls 5.0 | avg duration 65.9s


,id,repo,resolved,patch_chars,tool_calls,duration_s
0,fastapi_15661,fastapi/fastapi,False,0,7,65.6
1,fastapi_15588,fastapi/fastapi,False,0,3,66.2


## 6. Package Submission Archive

In [10]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')

Created /kaggle/working/submission.zip (0.17 MiB)
